# Importância de features bancárias: Bankai Extra Trees com e sem histograma

Este notebook compara somente duas configurações do Bankai Random Forest do PyPI: `extra_trees=True` com `max_bins=None` (sem histograma) e com `max_bins=63` (com histograma). Ambas usam os mesmos dados, parâmetros e seeds em cada repetição.

O ajuste usa 80% das linhas e reporta tempo de treino, ROC-AUC, average precision preditiva e qualidade do ranking das features informativas. O modo completo usa 500.000 linhas, 500 features, 100 árvores e 10 medições por configuração/cenário. Use `LIGHT_MODE=True` para validar rapidamente o fluxo.


## 1. Instalar dependências

In [1]:
%pip install bankai-random-forest==0.1.0a12 scikit-learn psutil threadpoolctl pandas numpy scipy


Reinicie o kernel se solicitado. As versões fixas ajudam outras pessoas a reproduzir o ambiente.

## 2. Recursos disponíveis

In [2]:
import os, platform, sys
import psutil
from threadpoolctl import threadpool_info

CPU_COUNT = os.cpu_count() or 1
physical = psutil.cpu_count(logical=False) or CPU_COUNT
memory = psutil.virtual_memory()
print({
    'machine': platform.machine(), 'platform': platform.platform(),
    'python': sys.version, 'logical_cpus': CPU_COUNT,
    'physical_cores': physical, 'memory_total_gib': round(memory.total / 2**30, 2),
    'memory_available_gib': round(memory.available / 2**30, 2),
    'threadpools_before_import': threadpool_info(),
})
N_JOBS = CPU_COUNT  # todos os núcleos lógicos disponíveis

{'machine': 'x86_64', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'logical_cpus': 44, 'physical_cores': 22, 'memory_total_gib': 172.95, 'memory_available_gib': 163.52, 'threadpools_before_import': [{'user_api': 'blas', 'internal_api': 'openblas', 'num_threads': 44, 'prefix': 'libscipy_openblas', 'filepath': '/usr/local/lib/python3.13/dist-packages/numpy.libs/libscipy_openblas64_-ff651d7f.so', 'version': '0.3.27', 'threading_layer': 'pthreads', 'architecture': 'SkylakeX'}]}


## 3. Configuração

In [9]:
from pathlib import Path

LIGHT_MODE = False  # use False para o benchmark completo
BASE_SEED = 20261012
N_WARMUP = 1
WORKDIR = Path.cwd().resolve()
REPO = WORKDIR.parent if WORKDIR.name == 'benchmarks' else WORKDIR
RESULT_DIR = REPO / 'benchmarks' / 'results-banking-feature-importance-bankai-extra-trees-histogram-pair-a12'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = RESULT_DIR / ('raw-light.csv' if LIGHT_MODE else 'raw-full.csv')
IMPORTANCE_PATH = RESULT_DIR / ('importance-light.csv' if LIGHT_MODE else 'importance-full.csv')

HISTOGRAM_BINS = 63
EXTRA_TREES = True
MODEL_NO_HIST = 'BankaiRF Extra Trees (sem histograma)'
MODEL_WITH_HIST = 'BankaiRF Extra Trees (max_bins=63)'
MODEL_NAMES = [MODEL_NO_HIST, MODEL_WITH_HIST]

if LIGHT_MODE:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 20_000, 100, 24
    N_REPEATS, N_TREES = 1, 20
else:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 500_000, 500, 24
    N_REPEATS, N_TREES = 10, 100

print({'light_mode': LIGHT_MODE, 'rows_per_scenario': N_ROWS,
       'features': N_FEATURES, 'informative_features': N_INFORMATIVE,
       'measured_repeats_per_model_scenario': N_REPEATS,
       'warmups_per_model_scenario': N_WARMUP, 'trees': N_TREES,
       'test_fraction': 0.20, 'n_jobs': N_JOBS,
       'histogram_bins': HISTOGRAM_BINS, 'extra_trees': EXTRA_TREES,
       'result_file': str(RAW_PATH)})


{'light_mode': False, 'rows_per_scenario': 500000, 'features': 500, 'informative_features': 24, 'measured_repeats_per_model_scenario': 10, 'warmups_per_model_scenario': 1, 'trees': 100, 'test_fraction': 0.2, 'n_jobs': 44, 'histogram_bins': 63, 'extra_trees': True, 'result_file': '/content/benchmarks/results-banking-feature-importance-bankai-extra-trees-histogram-pair-a12/raw-full.csv'}


### Recursos e duração

Modo completo: 4 cenários × 2 configurações × 10 medições + 8 aquecimentos = 88 ajustes. A matriz float32 de 500.000 × 500 usa cerca de 0,93 GiB; treino e holdout geram cópias adicionais. Confira a memória disponível antes de executar o modo completo.

As duas configurações usam Bankai com `extra_trees=True` e todos os CPUs lógicos detectados. Em cada repetição, dados, split e seed do modelo são iguais; a ordem entre as configurações alterna para reduzir viés de tempo.


## 4. Cenários e premissas

1. **Inadimplência de crédito (1,5% positivos):** prever atraso grave futuro; sinais de comprometimento de renda, uso de crédito, histórico, emprego e idade de crédito. Sinais fracos e correlações motivam árvores moderadas e folhas mínimas maiores.
2. **Fraude em transações (0,2% positivos):** valor atípico, horário/dispositivo incomum, velocidade e interações. Evento muito raro motiva árvores um pouco mais profundas para capturar combinações, mantendo folhas mínimas para limitar sobreajuste.
3. **Churn (6% positivos):** queda de atividade e saldo, reclamações, tarifas e recência de contato. Relações por segmento justificam profundidade intermediária.
4. **Aceitação de oferta (18% positivos):** capacidade financeira, engajamento, adequação e canal. Mais positivos e sinais relativamente diretos permitem árvores mais rasas.

Cada cenário tem 24 features informativas nos índices 0–23 e ruído nas restantes. Há proxies correlacionadas, limiares, curvas e interações. Nenhuma coluna revela o rótulo diretamente. São hipóteses sintéticas controladas, não retratos de clientes reais.

As escolhas iniciais de regularização estão na tabela `SCENARIOS`: folhas maiores e profundidade moderada para crédito raro; mais profundidade com folhas mínimas para fraude; valores intermediários para churn; árvores rasas para aceitação. São premissas de partida, não resultados de busca de hiperparâmetros.

As duas configurações Bankai usam 80% de amostras por árvore, `max_features='sqrt'`, profundidade e folha derivados da tabela, pesos de classe balanceados, `extra_trees=True` e `extra_seed=6`. A única diferença configurada é `max_bins`: `None` desativa histogramas; `63` usa histograma com 63 bins.

| Cenário | max_depth | min_samples_leaf | amostra de linhas |
|---|---:|---:|---:|
| Inadimplência | 8 | 500 | 80% |
| Fraude | 9 | 750 | 80% |
| Churn | 7 | 250 | 80% |
| Aceitação | 6 | 300 | 80% |

A folha mínima maior nos cenários com menos positivos reduz folhas sustentadas por poucos eventos.


## 5. Importações e confirmação do Bankai do PyPI

In [10]:
import gc, json, math, os, time, warnings
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from threadpoolctl import threadpool_limits
import bankai_random_forest
from bankai_random_forest import BankaiRandomForestClassifier

repo = REPO
package_file = Path(bankai_random_forest.__file__).resolve()
extension_file = Path(bankai_random_forest._core.__file__).resolve()
dist = metadata.distribution('bankai-random-forest')
EXPECTED_BANKAI_VERSION = '0.1.0a12'
if dist.version != EXPECTED_BANKAI_VERSION:
    raise RuntimeError(f'Esperado Bankai {EXPECTED_BANKAI_VERSION} do PyPI, encontrado {dist.version}')
direct_url = dist.read_text('direct_url.json')
python_source = (repo / 'python').resolve()
if package_file.is_relative_to(python_source) or extension_file.is_relative_to(python_source):
    raise RuntimeError(f'Bankai foi importado do código-fonte local, esperado PyPI: {package_file}')
if direct_url:
    install_origin = json.loads(direct_url)
    if install_origin.get('dir_info', {}).get('editable', False) or install_origin.get('url', '').startswith('file:'):
        raise RuntimeError('Bankai foi instalado de um checkout/arquivo local, esperado PyPI.')
print({'bankai': dist.version, 'package': str(package_file),
       'native_extension': str(extension_file),
       'scikit_learn': metadata.version('scikit-learn')})


{'bankai': '0.1.0a12', 'package': '/usr/local/lib/python3.13/dist-packages/bankai_random_forest/__init__.py', 'native_extension': '/usr/local/lib/python3.13/dist-packages/bankai_random_forest/_core.abi3.so', 'scikit_learn': '1.9.1'}


## 6. Geradores dos datasets

As features 0–23 são informativas por construção. A tabela `SCENARIOS` registra prevalência e parâmetros assumidos para cada problema. Cada par de configurações compartilha seed de dados, split e seed do modelo. Os resultados são salvos em CSV para permitir retomada e análise posterior.


In [11]:
SCENARIOS = {
    'credit_default': {'prevalence': 0.015, 'description': 'Inadimplência futura', 'max_depth': 8, 'min_leaf': 500, 'row_fraction': 0.80},
    'transaction_fraud': {'prevalence': 0.002, 'description': 'Fraude confirmada', 'max_depth': 9, 'min_leaf': 750, 'row_fraction': 0.80},
    'customer_churn': {'prevalence': 0.060, 'description': 'Encerramento futuro', 'max_depth': 7, 'min_leaf': 250, 'row_fraction': 0.80},
    'offer_acceptance': {'prevalence': 0.180, 'description': 'Aceitação de oferta', 'max_depth': 6, 'min_leaf': 300, 'row_fraction': 0.80},
}

def calibrate_intercept(signal, prevalence):
    signal = (signal - signal.mean()) / max(signal.std(), 1e-8)
    lo, hi = -30.0, 30.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if expit(signal + mid).mean() > prevalence:
            hi = mid
        else:
            lo = mid
    return signal + (lo + hi) / 2

def make_dataset(scenario, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(N_ROWS, N_FEATURES)).astype(np.float32)
    # Indicadores e variáveis monetárias com assimetria, comuns em dados bancários.
    X[:, 4:8] = rng.binomial(1, [0.12, 0.08, 0.18, 0.35], size=(N_ROWS, 4)).astype(np.float32)
    X[:, 8:12] = np.log1p(rng.lognormal(mean=2.0, sigma=0.8, size=(N_ROWS, 4))).astype(np.float32)
    X[:, 8:12] = (X[:, 8:12] - X[:, 8:12].mean(axis=0)) / (X[:, 8:12].std(axis=0) + 1e-6)
    # Proxies correlacionadas; todas continuam informativas para a verdade sintética.
    X[:, 16:20] = 0.75 * X[:, 0:4] + 0.66 * rng.normal(size=(N_ROWS, 4)).astype(np.float32)
    z = X

    if scenario == 'credit_default':
        signal = (1.0*z[:,0] + .85*z[:,1] + .65*z[:,2] + .5*z[:,3]
                  + .8*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6] - .4*z[:,7]
                  + .65*z[:,8] - .55*z[:,9] + .45*z[:,10] + .35*z[:,11]
                  + .7*np.maximum(z[:,12]-.8, 0) + .6*(z[:,13]**2-1)
                  + .65*z[:,14]*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'transaction_fraud':
        signal = (1.0*z[:,0] + .8*np.maximum(z[:,1]-1, 0) + .7*(z[:,2] > 1.2)
                  + .9*z[:,3]*z[:,4] + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .8*np.abs(z[:,7]) + .55*z[:,8] + .5*z[:,9]
                  + .65*(z[:,10]**2-1) + .7*z[:,11]*z[:,12]
                  + .5*z[:,13] + .4*z[:,14] + .35*z[:,15]
                  + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'customer_churn':
        signal = (-.9*z[:,0] - .8*z[:,1] + .7*z[:,2] + .6*z[:,3]
                  + .65*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .45*z[:,7] - .65*z[:,8] + .55*z[:,9] + .45*z[:,10]
                  + .6*np.maximum(z[:,11]-.7, 0) + .65*z[:,12]*z[:,13]
                  + .5*np.tanh(z[:,14]) + .4*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'offer_acceptance':
        signal = (.8*z[:,0] + .7*z[:,1] - .5*z[:,2] + .65*z[:,3]
                  + .6*(z[:,4] > .5) + .45*(z[:,5] > .5) + .5*z[:,6]
                  + .4*z[:,7] + .75*z[:,8] + .55*z[:,9] - .4*z[:,10]
                  + .6*(z[:,11]**2-1) + .65*z[:,12]*z[:,13]
                  + .5*np.tanh(z[:,14]) + .35*z[:,15] + .18*z[:,16:24].sum(axis=1))
    else:
        raise ValueError(f'Cenário desconhecido: {scenario}')
    logits = calibrate_intercept(signal, SCENARIOS[scenario]['prevalence'])
    y = rng.binomial(1, expit(logits)).astype(np.int8)
    return X, y, np.arange(N_FEATURES) < N_INFORMATIVE

print({name: info for name, info in SCENARIOS.items()})

{'credit_default': {'prevalence': 0.015, 'description': 'Inadimplência futura', 'max_depth': 8, 'min_leaf': 500, 'row_fraction': 0.8}, 'transaction_fraud': {'prevalence': 0.002, 'description': 'Fraude confirmada', 'max_depth': 9, 'min_leaf': 750, 'row_fraction': 0.8}, 'customer_churn': {'prevalence': 0.06, 'description': 'Encerramento futuro', 'max_depth': 7, 'min_leaf': 250, 'row_fraction': 0.8}, 'offer_acceptance': {'prevalence': 0.18, 'description': 'Aceitação de oferta', 'max_depth': 6, 'min_leaf': 300, 'row_fraction': 0.8}}


## 7. Configurações do Bankai e cálculo de importâncias


In [12]:
def estimator_factory(seed, scenario, model_name):
    cfg = SCENARIOS[scenario]
    max_bins = None if model_name == MODEL_NO_HIST else HISTOGRAM_BINS
    return BankaiRandomForestClassifier(
        n_estimators=N_TREES,
        max_depth=cfg['max_depth'],
        min_samples_leaf=cfg['min_leaf'],
        max_features='sqrt',
        max_samples=cfg['row_fraction'],
        bootstrap=True,
        max_bins=max_bins,
        importance_type='gain',
        class_weight='balanced',
        extra_trees=EXTRA_TREES,
        extra_seed=6,
        random_state=int(seed % (2**31 - 1)),
        n_jobs=N_JOBS,
    )


## 8. Aquecimento, holdout, medições e retomada


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
TOP_K = N_INFORMATIVE

def evaluate_one(X, y, informative, scenario, model_name, repeat_no, data_seed, model_seed):
    indices = np.arange(len(y))
    tr, te = train_test_split(indices, test_size=0.20, stratify=y,
                              random_state=int((data_seed+73) % (2**31-1)))
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
    model = estimator_factory(model_seed, scenario, model_name)
    t=time.perf_counter()
    with threadpool_limits(limits=N_JOBS): model.fit(Xtr,ytr)
    fit_s=time.perf_counter()-t
    t=time.perf_counter()
    with threadpool_limits(limits=N_JOBS): prob=model.predict_proba(Xte)[:,1]
    predict_s=time.perf_counter()-t
    imp=np.nan_to_num(np.asarray(model.feature_importances_,dtype=np.float64),nan=0.0,posinf=0.0,neginf=0.0)
    ranking=np.argsort(-imp,kind='stable'); rank_position=np.empty_like(ranking)
    rank_position[ranking]=np.arange(1,len(ranking)+1); top=ranking[:TOP_K]
    tp=int(informative[top].sum())
    max_bins = None if model_name == MODEL_NO_HIST else HISTOGRAM_BINS
    row={'scenario':scenario,'model':model_name,'repeat':repeat_no,'data_seed':data_seed,
         'model_seed':model_seed,'fit_seconds':fit_s,'prediction_seconds':predict_s,
         'feature_average_precision':float(average_precision_score(informative.astype(int),imp)),
         'recall_at_k':tp/int(informative.sum()),'precision_at_k':tp/max(len(top),1),'top_k':TOP_K,
         'roc_auc':float(roc_auc_score(yte,prob)),'pr_auc':float(average_precision_score(yte,prob)),
         'positive_prevalence':float(y.mean()),'train_rows':len(tr),'test_rows':len(te),
         'features':X.shape[1],'informative_features':int(informative.sum()),'n_jobs':N_JOBS,
         'trees':N_TREES,'max_depth':SCENARIOS[scenario]['max_depth'],
         'min_samples_leaf':SCENARIOS[scenario]['min_leaf'],'row_fraction':SCENARIOS[scenario]['row_fraction'],
         'bankai_max_bins':max_bins,'bankai_extra_trees':EXTRA_TREES,
         'bankai_extra_seed':6,'run_signature':RUN_SIGNATURE}
    scores=[{'scenario':scenario,'model':model_name,'repeat':repeat_no,
             'data_seed':data_seed,'feature_index':int(j),'importance':float(imp[j]),
             'rank':int(rank_position[j]),'is_informative':bool(informative[j]),
             'run_signature':RUN_SIGNATURE} for j in range(len(imp))]
    del model,Xtr,Xte,ytr,yte,prob
    return row,scores

RUN_SIGNATURE=json.dumps({'light_mode':LIGHT_MODE,'rows':N_ROWS,'features':N_FEATURES,
 'repeats':N_REPEATS,'trees':N_TREES,'base_seed':BASE_SEED,'n_jobs':N_JOBS,
 'histogram_bins':HISTOGRAM_BINS,'extra_trees':EXTRA_TREES,'extra_seed':6,
 'scenarios':SCENARIOS,'models':MODEL_NAMES,'top_k':TOP_K},sort_keys=True)
records=pd.read_csv(RAW_PATH).to_dict('records') if RAW_PATH.exists() else []
score_records=pd.read_csv(IMPORTANCE_PATH).to_dict('records') if IMPORTANCE_PATH.exists() else []
if any(r.get('run_signature')!=RUN_SIGNATURE for r in records+score_records):
    raise RuntimeError('CSVs existentes usam outra configuração; use uma pasta limpa.')
done={(r['scenario'],r['model'],int(r['repeat'])) for r in records if int(r['repeat'])>=0}

def save_checkpoint(frame, path):
    tmp=path.with_name(path.name + '.tmp')
    frame.to_csv(tmp,index=False)
    tmp.replace(path)

for si,scenario in enumerate(SCENARIOS):
    print(); print(f'=== {scenario} ===',flush=True)
    warm_seed=BASE_SEED+si*1_000_000+777
    Xw,yw,_=make_dataset(scenario,warm_seed)
    tr,_=train_test_split(np.arange(len(yw)),test_size=.2,stratify=yw,
                          random_state=(warm_seed+73)%(2**31-1))
    for name in MODEL_NAMES:
        model=estimator_factory(warm_seed+800,scenario,name)
        with threadpool_limits(limits=N_JOBS): model.fit(Xw[tr],yw[tr])
        del model; gc.collect()
        print('warmup complete:',name,flush=True)
    del Xw,yw,tr; gc.collect()
    for rep in range(N_REPEATS):
        if all((scenario,name,rep) in done for name in MODEL_NAMES): continue
        data_seed=BASE_SEED+si*1_000_000+rep*10_000+500
        model_seed=BASE_SEED+si*1_000_000+rep*10_000+900
        X,y,informative=make_dataset(scenario,data_seed)
        ordered_names=MODEL_NAMES.copy()
        if rep % 2: ordered_names.reverse()
        for name in ordered_names:
            if (scenario,name,rep) in done: continue
            print(f'{scenario} | {rep+1}/{N_REPEATS} | {name}',flush=True)
            row,scores=evaluate_one(X,y,informative,scenario,name,rep,data_seed,model_seed)
            records.append(row); score_records.extend(scores)
            score_frame=(pd.DataFrame(score_records)
                .drop_duplicates(['scenario','model','repeat','feature_index'],keep='last'))
            save_checkpoint(score_frame,IMPORTANCE_PATH)
            save_checkpoint(pd.DataFrame(records),RAW_PATH)
            done.add((scenario,name,rep))
            print({k:row[k] for k in ('fit_seconds','feature_average_precision','recall_at_k',
                                      'precision_at_k','roc_auc','pr_auc')},flush=True)
        del X,y; gc.collect()
print('Metrics:',RAW_PATH); print('Per-feature importance:',IMPORTANCE_PATH)


=== credit_default ===
warmup complete: BankaiRF Extra Trees (sem histograma)
warmup complete: BankaiRF Extra Trees (max_bins=63)
credit_default | 1/10 | BankaiRF Extra Trees (sem histograma)
{'fit_seconds': 7.927421858003072, 'feature_average_precision': 0.8933971380110808, 'recall_at_k': 0.875, 'precision_at_k': 0.875, 'roc_auc': 0.7118525379971348, 'pr_auc': 0.04109847243010165}
credit_default | 1/10 | BankaiRF Extra Trees (max_bins=63)
{'fit_seconds': 4.583146763994591, 'feature_average_precision': 0.9314690752455677, 'recall_at_k': 0.9166666666666666, 'precision_at_k': 0.9166666666666666, 'roc_auc': 0.703809693547395, 'pr_auc': 0.03586877077709715}
credit_default | 2/10 | BankaiRF Extra Trees (max_bins=63)
{'fit_seconds': 4.543427513999632, 'feature_average_precision': 0.8942464402805439, 'recall_at_k': 0.8333333333333334, 'precision_at_k': 0.8333333333333334, 'roc_auc': 0.7154902716752866, 'pr_auc': 0.03941306562083287}
credit_default | 2/10 | BankaiRF Extra Trees (sem histogram

## 9. Resumo por cenário


In [14]:
raw=pd.read_csv(RAW_PATH)
summary=(raw.groupby(['scenario','model'],sort=False)
 .agg(runs=('repeat','nunique'),n_jobs=('n_jobs','first'),bankai_max_bins=('bankai_max_bins','first'),
      max_depth=('max_depth','first'),min_samples_leaf=('min_samples_leaf','first'),
      row_fraction=('row_fraction','first'),fit_median_s=('fit_seconds','median'),
      fit_iqr_s=('fit_seconds',lambda x:x.quantile(.75)-x.quantile(.25)),
      feature_ap_median=('feature_average_precision','median'),
      recall_at_k_median=('recall_at_k','median'),precision_at_k_median=('precision_at_k','median'),
      roc_auc_median=('roc_auc','median'),pr_auc_median=('pr_auc','median')).reset_index())
summary.to_csv(RESULT_DIR/('summary-light.csv' if LIGHT_MODE else 'summary-full.csv'),index=False)
display(summary)

# Comparação pareada: mesmos cenário, repetição e seeds nos dois modos.
pair=(raw.pivot(index=['scenario','repeat','data_seed','model_seed'],columns='model',
                 values=['fit_seconds','feature_average_precision','recall_at_k','precision_at_k','roc_auc','pr_auc'])
      .dropna().reset_index())
pair.columns=['_'.join(str(part) for part in col if str(part)) if isinstance(col,tuple) else col
              for col in pair.columns]
pair['fit_time_ratio_hist_over_no_hist']=pair[f'fit_seconds_{MODEL_WITH_HIST}']/pair[f'fit_seconds_{MODEL_NO_HIST}']
pair['fit_time_delta_hist_minus_no_hist_s']=pair[f'fit_seconds_{MODEL_WITH_HIST}']-pair[f'fit_seconds_{MODEL_NO_HIST}']
pair['feature_ap_delta_hist_minus_no_hist']=pair[f'feature_average_precision_{MODEL_WITH_HIST}']-pair[f'feature_average_precision_{MODEL_NO_HIST}']
pair.to_csv(RESULT_DIR/('paired-light.csv' if LIGHT_MODE else 'paired-full.csv'),index=False)
pair_summary=pair.groupby('scenario',sort=False).agg(
    paired_runs=('repeat','nunique'),fit_time_ratio_median=('fit_time_ratio_hist_over_no_hist','median'),
    fit_time_delta_median_s=('fit_time_delta_hist_minus_no_hist_s','median'),
    feature_ap_delta_median=('feature_ap_delta_hist_minus_no_hist','median')).reset_index()
pair_summary.to_csv(RESULT_DIR/('paired-summary-light.csv' if LIGHT_MODE else 'paired-summary-full.csv'),index=False)
print('Comparação pareada: razão < 1 indica que o histograma foi mais rápido.')
display(pair_summary)
print('Resultados:',RESULT_DIR)


,scenario,model,runs,n_jobs,bankai_max_bins,max_depth,min_samples_leaf,row_fraction,fit_median_s,fit_iqr_s,feature_ap_median,recall_at_k_median,precision_at_k_median,roc_auc_median,pr_auc_median
0,credit_default,BankaiRF Extra Trees (sem histograma),10,44,NaN,8,500,0.8,7.985334,0.045751,0.923108,0.875000,0.875000,0.709995,0.039973
1,credit_default,BankaiRF Extra Trees (max_bins=63),10,44,63.0,8,500,0.8,4.567641,0.052887,0.920019,0.875000,0.875000,0.710455,0.038259
2,transaction_fraud,BankaiRF Extra Trees (sem histograma),10,44,NaN,9,750,0.8,8.476621,0.075504,0.624380,0.541667,0.541667,0.643062,0.004043
3,transaction_fraud,BankaiRF Extra Trees (max_bins=63),10,44,63.0,9,750,0.8,4.810295,0.060820,0.593607,0.541667,0.541667,0.637428,0.003756
4,customer_churn,BankaiRF Extra Trees (sem histograma),10,44,NaN,7,250,0.8,7.383218,0.053686,0.986752,0.958333,0.958333,0.695573,0.131897
5,customer_churn,BankaiRF Extra Trees (max_bins=63),10,44,63.0,7,250,0.8,4.199622,0.117254,0.993519,0.958333,0.958333,0.712728,0.143241
6,offer_acceptance,BankaiRF Extra Trees (sem histograma),10,44,NaN,6,300,0.8,6.591646,0.058597,0.970432,0.958333,0.958333,0.681201,0.314403
7,offer_acceptance,BankaiRF Extra Trees (max_bins=63),10,44,63.0,6,300,0.8,3.840546,0.067074,0.983259,0.958333,0.958333,0.697495,0.334575


Comparação pareada: razão < 1 indica que o histograma foi mais rápido.


,scenario,paired_runs,fit_time_ratio_median,fit_time_delta_median_s,feature_ap_delta_median
0,credit_default,10,0.572783,-3.429812,-0.002981
1,customer_churn,10,0.568764,-3.184169,0.001406
2,offer_acceptance,10,0.581868,-2.735725,-0.005047
3,transaction_fraud,10,0.566923,-3.660617,-0.006672


Resultados: /content/benchmarks/results-banking-feature-importance-bankai-extra-trees-histogram-pair-a12


## 10. Interpretação

A tabela de resumo apresenta as medianas por cenário e configuração. A tabela pareada compara cada repetição com os mesmos dados e seeds; `fit_time_ratio_median` abaixo de 1 indica que a configuração com histograma foi mais rápida. A diferença de average precision mostra como o histograma afetou o ranking das features informativas.

Os CSVs guardam métricas por repetição e a importância de cada feature. Para o benchmark completo, altere `LIGHT_MODE` para `False` na seção de configuração e execute novamente as células a partir dali.

Para uso bancário real, valide em períodos futuros, investigue leakage e estabilidade por segmentos. Importância preditiva não demonstra causalidade.
